# Fine-tune Nemotron 3.5 ASR on CDLI Non-Standard Kenyan Speech (Noise-Robust)

**This notebook is a from-scratch rewrite, not a port.** `nvidia/nemotron-3.5-asr-streaming-0.6b` is architecturally unlike Whisper:

| | Whisper | Nemotron 3.5 ASR |
|---|---|---|
| Architecture | Encoder-decoder seq2seq | Cache-Aware FastConformer encoder + RNNT decoder + language-ID prompt |
| Fine-tuning API | `transformers.Seq2SeqTrainer` | NVIDIA NeMo (`speech_to_text_finetune.py`), Hydra/YAML configs |
| Data format | HF `datasets` + feature extractor | JSON-lines manifest (`audio_filepath`, `duration`, `text`, `lang`, `target_lang`) |
| Loss | Cross-entropy over decoder tokens | Transducer (RNNT) loss |
| Latency control | N/A | `att_context_size` (streaming chunk size) |

The Hugging Face `transformers` integration for this model (`AutoModelForRNNT`/`AutoProcessor`, available in `transformers>=5.13`) is inference/streaming-only. NVIDIA's own recipe for fine-tuning uses NeMo directly. This notebook follows that recipe:

- Blog: https://huggingface.co/blog/nvidia/fine-tuning-nemotron-35-asr
- Reference notebook: https://github.com/nvidia-riva/tutorials/blob/main/asr-finetune-nemotron-3.5-asr-streaming-prompt.ipynb

**What carries over from your Whisper notebook, in spirit:**
- Same dataset (`cdli/kenyan_english_nonstandard_speech_v1.0`), same `audio` / `transcription` / `audio_length` columns.
- Same noise-robustness methodology: report clean WER, noisy WER, and the gap between them.
- Same settings-first structure so you can tweak one cell per run.

**What's genuinely different and can't be preserved:**
- No encoder/decoder freezing scheme (RNNT fine-tuning is a full-model `init_from_nemo_model` warm start, controlled via YAML, not `requires_grad` flags).
- Waveform augmentation during *training* is now done by NeMo's built-in on-the-fly `augmentor` (more efficient than pre-baking), not your custom `augment_audio()` call inside `.map()`. Your `augment_audio()` function is kept, but repurposed to build a **static noisy eval set** (dev/test), which is what actually produced your clean-vs-noisy WER gap numbers before.
- Training runs as a NeMo/Hydra script invocation (subprocess), not a Python training loop, because RNNT loss and the streaming cache-aware encoder are implemented inside NeMo's `EncDecRNNTBPEModel` / PTL `Trainer`, not exposed as a plain `nn.Module.forward()` you'd wire into `Seq2SeqTrainer`.

**One judgment call to confirm before you run this:** language tag. Kenyan English isn't its own locale in the model's 40 supported language-locales; this notebook defaults to `en-GB` (`LANG_TAG` below) since Kenyan English orthography and vocabulary lean British (colour, programme, etc.). Change to `en-US` if your transcripts follow American conventions instead.

## Authentication

In [1]:
from huggingface_hub import login
HF_TOKEN = input("Enter HF token: ")
login(token=HF_TOKEN)


## Install Dependencies

NeMo's ASR collection plus the system audio tools it shells out to (`sox`, `ffmpeg`, `libsndfile`). This mirrors the setup step in NVIDIA's reference notebook.

In [2]:
!apt-get update -qq && apt-get install -y -qq sox libsndfile1 ffmpeg libsox-fmt-mp3 jq > /dev/null
!pip install -q "nemo_toolkit[asr]" torchcodec text-unidecode Cython soundfile librosa jiwer evaluate huggingface_hub omegaconf hydra-core



[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


### Directories

In [3]:
import os

LOCAL_STORAGE_DIR = '/jupyter_kernel'
BASE_DIR = os.path.join(LOCAL_STORAGE_DIR, 'trained_models')
os.makedirs(BASE_DIR, exist_ok=True)

# Set run name -- increment for each new run
RUN_NAME = 'nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1'
OUTPUT_DIR = os.path.join(BASE_DIR, RUN_NAME)

# Where converted audio + manifests live (separate from the model output dir)
DATA_DIR = os.path.join(LOCAL_STORAGE_DIR, 'nemo_data', RUN_NAME)
os.makedirs(DATA_DIR, exist_ok=True)

# Where the NeMo repo (for training/eval scripts + base configs) is cloned
NEMO_DIR = os.path.join(LOCAL_STORAGE_DIR, 'NeMo')

print(f"Will write model to: {OUTPUT_DIR}")
print(f"Will write manifests/wavs to: {DATA_DIR}")
print(f"Will clone NeMo into: {NEMO_DIR}")
if os.path.exists(OUTPUT_DIR):
    raise ValueError("Output directory already exists. Increment run number or delete existing directory.")


Will write model to: /jupyter_kernel/trained_models/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1
Will write manifests/wavs to: /jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1
Will clone NeMo into: /jupyter_kernel/NeMo


### Model and Dataset Settings

In [4]:
MODEL_ID = "nvidia/nemotron-3.5-asr-streaming-0.6b"
NEMO_CHECKPOINT_FILENAME = "nemotron-3.5-asr-streaming-0.6b.nemo"  # base checkpoint file on the HF repo

DATASET_NAME = "cdli/kenyan_english_nonstandard_speech_v1.0"

# See the note in the intro cell -- confirm this matches your transcript conventions.
LANG_TAG = "en-GB"     # used for both the "lang" and "target_lang" manifest fields
TARGET_LANG_AT_INFERENCE = LANG_TAG  # set to "auto" instead if you want language-ID behavior at eval time

print(f"Base model: {MODEL_ID} | Language tag: {LANG_TAG} | Dataset: {DATASET_NAME}")


Base model: nvidia/nemotron-3.5-asr-streaming-0.6b | Language tag: en-GB | Dataset: cdli/kenyan_english_nonstandard_speech_v1.0


### Augmentation Settings

Two separate uses of augmentation, matching your original noise-robustness methodology:

1. **Training-time augmentation** now happens inside NeMo's dataloader via an `augmentor` config block (on-the-fly, every epoch, no pre-baked copies needed). Built from the flags below.
2. **Static noisy eval set** — a one-time noisy copy of dev/test, built with your original `augment_audio()` waveform function, so clean-vs-noisy WER stays comparable to your Whisper runs.

In [5]:
# Master switch: set to False to disable all waveform augmentation (reproduces a clean-only baseline)
USE_WAVEFORM_AUGMENTATION = True

# Probability that any single augmentation is applied to a given example (used for the static noisy eval set)
AUGMENT_PROB = 0.4

# Individual augmentation enable flags
AUG_VOLUME_PERTURB = True   # always mild, safe to keep on
AUG_GAUSSIAN_NOISE  = True   # simulates crowd/ambient noise
AUG_GSM_CODEC       = True   # simulates mobile network compression
AUG_REVERB          = False  # simulates small room/kiosk acoustics

# Noise intensity bounds (increase cautiously for larger datasets)
NOISE_LEVEL_MIN = 0.002
NOISE_LEVEL_MAX = 0.01

# NeMo's built-in AudioAugmentor config for TRAINING only. Probabilities here are NeMo's own
# per-perturbation "prob" field (independent of AUGMENT_PROB, which only drives the static eval set below).
NEMO_AUGMENTOR_CONFIG = {}
if USE_WAVEFORM_AUGMENTATION:
    if AUG_VOLUME_PERTURB:
        NEMO_AUGMENTOR_CONFIG["gain"] = {"prob": 1.0, "min_gain_dbfs": -6.0, "max_gain_dbfs": 6.0}
    if AUG_GAUSSIAN_NOISE:
        NEMO_AUGMENTOR_CONFIG["white_noise"] = {
            "prob": 0.5,
            "min_level": -90,  # dB, tune against NOISE_LEVEL_MIN/MAX if you recalibrate
            "max_level": -46,
        }
    if AUG_GSM_CODEC:
        # Simulates mobile-network compression artifacts via a codec round-trip (ffmpeg-backed).
        NEMO_AUGMENTOR_CONFIG["transcode_aug"] = {"prob": 0.4, "codecs": ["g722", "amr-nb"]}
    if AUG_REVERB:
        # Requires a directory of impulse response (RIR) wav files -- point this at one if you enable reverb.
        NEMO_AUGMENTOR_CONFIG["impulse"] = {"prob": 0.2, "manifest_path": None}

print("NeMo training-time augmentor config:", NEMO_AUGMENTOR_CONFIG or "(disabled)")


NeMo training-time augmentor config: {'gain': {'prob': 1.0, 'min_gain_dbfs': -6.0, 'max_gain_dbfs': 6.0}, 'white_noise': {'prob': 0.5, 'min_level': -90, 'max_level': -46}, 'transcode_aug': {'prob': 0.4, 'codecs': ['g722', 'amr-nb']}}


### Training Settings

Mapped from your Whisper trainer args onto NeMo/Hydra config overrides. A few notes:
- `att_context_size` controls streaming latency (encoder look-ahead). `[70,13]` is a reasonable default full-context-ish setting for offline fine-tuning; NVIDIA's chunk presets (80/160/320/560/1120ms) map to specific `att_context_size` values if you need a specific streaming latency at deployment time.
- NeMo's early stopping/checkpointing is configured through `exp_manager`, monitoring `val_wer` (the validation WER NeMo computes itself each eval epoch), not through `Seq2SeqTrainer` callbacks.

In [6]:
MAX_EPOCHS = 10
MAX_STEPS  = -1     # -1 means "let max_epochs govern"; set a positive int to cap steps instead

LEARNING_RATE   = 5e-5     # RNNT fine-tuning typically wants a higher LR than Whisper cross-entropy fine-tuning
LR_WARMUP_STEPS = 100
WEIGHT_DECAY    = 0.01

BATCH_SIZE      = 8
EVAL_BATCH_SIZE = 8

EARLY_STOPPING_PATIENCE = 7
NUM_CHECKPOINTS_TO_STORE = 2

USE_FP16 = False
USE_BF16 = True   # enable for A100/A40
PRECISION = "bf16" if USE_BF16 else (16 if USE_FP16 else 32)

# Streaming chunk / look-ahead config, in encoder-frame units. [70,13] ~ a large-context setting
# suited to fine-tuning; tighten this later if you need a specific low-latency streaming profile.
ATT_CONTEXT_SIZE = [70, 13]

LIMIT_TO_30_SECONDS = True  # dataset-parity filter carried over from the Whisper notebook

print(f"Epochs: {MAX_EPOCHS} | LR: {LEARNING_RATE} | Batch size: {BATCH_SIZE} | Precision: {PRECISION}")


Epochs: 10 | LR: 5e-05 | Batch size: 8 | Precision: bf16


## Imports and Environment Setup

In [7]:
import glob
import json
import os
import random
import subprocess
import sys

import datasets
import evaluate
import librosa
import numpy as np
import soundfile as sf
import torch
import torchaudio.transforms as T
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf

# Disable dataset caching (saves disk on Modal volumes)
datasets.disable_caching()
print('Dataset caching:', datasets.is_caching_enabled())

torch.set_num_threads(1)

wer_metric = evaluate.load("wer")


Dataset caching: False


## Waveform Augmentation (for the static noisy eval set)

Unchanged from your Whisper notebook. Used once, to build a noisy copy of the dev/test audio for robustness reporting -- **not** used during training anymore (training augmentation is handled by NeMo's on-the-fly `augmentor`, configured above).

In [8]:
def augment_audio(audio_array: np.ndarray, sample_rate: int, apply_prob: float = 0.5) -> np.ndarray:
    """
    Apply waveform-level augmentations to simulate Kenyan deployment conditions.

    Args:
        audio_array: Raw audio waveform as numpy array.
        sample_rate: Audio sample rate (typically 16000 Hz).
        apply_prob: Probability that each stochastic augmentation is applied.

    Returns:
        Augmented waveform as numpy array, same shape as input.
    """
    waveform = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0)  # (1, T)

    # 1. Volume perturbation (always applied, mild range)
    if AUG_VOLUME_PERTURB:
        gain = random.uniform(0.7, 1.3)
        waveform = waveform * gain

    # 2. Gaussian noise (crowd/ambient)
    if AUG_GAUSSIAN_NOISE and random.random() < apply_prob:
        noise_level = random.uniform(NOISE_LEVEL_MIN, NOISE_LEVEL_MAX)
        noise = torch.randn_like(waveform) * noise_level
        waveform = waveform + noise

    # 3. GSM codec simulation (mobile network compression artifact)
    # Downsample to 8kHz then resample back to original rate
    if AUG_GSM_CODEC and random.random() < apply_prob:
        resample_down = T.Resample(orig_freq=sample_rate, new_freq=8000)
        resample_up   = T.Resample(orig_freq=8000, new_freq=sample_rate)
        waveform = resample_up(resample_down(waveform))

    # 4. Room reverb (small kiosk/office acoustic echo)
    # Implemented as a short delay blend; lower prob since it's more disruptive
    if AUG_REVERB and random.random() < apply_prob * 0.5:
        reverb_gain   = random.uniform(0.1, 0.25)
        delay_samples = random.randint(int(0.01 * sample_rate), int(0.05 * sample_rate))
        delayed = torch.zeros_like(waveform)
        delayed[:, delay_samples:] = waveform[:, :-delay_samples]
        waveform = waveform + reverb_gain * delayed

    # Clip to prevent saturation from compounded augmentations
    waveform = torch.clamp(waveform, -1.0, 1.0)
    return waveform.squeeze(0).numpy()


## Clone and Set Up NeMo

We clone the NeMo repo (rather than relying only on the pip package) because the fine-tuning and streaming-eval scripts, plus the base YAML config for this model family, live under `examples/asr/` and aren't shipped as importable package data.

In [9]:
BRANCH = 'main'  # subject to change to a fixed tag/version upon release
if not os.path.exists(NEMO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/NVIDIA-NeMo/NeMo", NEMO_DIR], check=True)

os.environ["PYTHONPATH"] = NEMO_DIR + ":" + os.environ.get("PYTHONPATH", "")

BASE_CONFIG_DIR = os.path.join(NEMO_DIR, "examples/asr/conf/fastconformer/cache_aware_streaming")
BASE_CONFIG_NAME = "fastconformer_transducer_bpe_streaming_prompt.yaml"
BASE_CONFIG_PATH = os.path.join(BASE_CONFIG_DIR, BASE_CONFIG_NAME)
assert os.path.exists(BASE_CONFIG_PATH), f"Expected base config at {BASE_CONFIG_PATH} -- NeMo repo layout may have changed."
print(f"Base config found at: {BASE_CONFIG_PATH}")


Cloning into '/jupyter_kernel/NeMo'...


Base config found at: /jupyter_kernel/NeMo/examples/asr/conf/fastconformer/cache_aware_streaming/fastconformer_transducer_bpe_streaming_prompt.yaml


Updating files: 100% (2306/2306), done.


## Download the Base Checkpoint

In [10]:
HF_CKPT = hf_hub_download(repo_id=MODEL_ID, filename=NEMO_CHECKPOINT_FILENAME)
print(f"Downloaded base .nemo checkpoint to: {HF_CKPT}")


Downloaded base .nemo checkpoint to: /root/.cache/huggingface/hub/models--nvidia--nemotron-3.5-asr-streaming-0.6b/snapshots/f3d333391852ba876df169dcc9ba902d25b6ab0b/nemotron-3.5-asr-streaming-0.6b.nemo


## Convert the CDLI Dataset to NeMo Manifests

NeMo expects audio on disk plus a JSON-lines manifest (`audio_filepath`, `duration`, `text`, `lang`, `target_lang`), not an in-memory HF `datasets` object. This step:
1. Loads each split of `cdli/kenyan_english_nonstandard_speech_v1.0` (same `audio` / `transcription` / `audio_length` columns as your Whisper notebook).
2. Writes each example's audio to a standalone `.wav` file.
3. Writes the corresponding manifest line.

In [11]:
def build_manifest_from_hf_split(dataset_name: str, split: str, wavs_dir: str, manifest_path: str,
                                  lang_tag: str, limit_to_30_seconds: bool = True):
    """Load an HF dataset split and export it to a NeMo-format manifest + wav files."""
    if split not in ['train', 'test', 'validation']:
        raise ValueError("split must be one of 'train', 'test', or 'validation'")

    ds = datasets.load_dataset(dataset_name, split=split, streaming=False)
    orig_len = len(ds)
    if limit_to_30_seconds:
        ds = ds.filter(lambda ex: ex['audio_length'] <= 30)
        print(f"[{split}] Filtered {orig_len} -> {len(ds)} examples (<= 30s)")

    os.makedirs(wavs_dir, exist_ok=True)
    with open(manifest_path, 'w') as fout:
        for i, example in enumerate(ds):
            audio_array = example['audio']['array']
            sr = example['audio']['sampling_rate']
            wav_path = os.path.join(wavs_dir, f"{split}_{i:06d}.wav")
            sf.write(wav_path, audio_array, sr)
            duration = len(audio_array) / sr

            metadata = {
                "audio_filepath": wav_path,
                "duration": duration,
                "text": example['transcription'],
                "lang": lang_tag,
                "target_lang": lang_tag,
            }
            fout.write(json.dumps(metadata) + "\n")

    print(f"[{split}] Wrote {i + 1} entries to {manifest_path}")
    return manifest_path


train_wavs_dir = os.path.join(DATA_DIR, 'wavs', 'train')
dev_wavs_dir   = os.path.join(DATA_DIR, 'wavs', 'validation')
test_wavs_dir  = os.path.join(DATA_DIR, 'wavs', 'test')

train_manifest = os.path.join(DATA_DIR, 'train_manifest.json')
dev_manifest   = os.path.join(DATA_DIR, 'dev_manifest.json')
test_manifest  = os.path.join(DATA_DIR, 'test_manifest.json')

build_manifest_from_hf_split(DATASET_NAME, 'train', train_wavs_dir, train_manifest, LANG_TAG, LIMIT_TO_30_SECONDS)
build_manifest_from_hf_split(DATASET_NAME, 'validation', dev_wavs_dir, dev_manifest, LANG_TAG, LIMIT_TO_30_SECONDS)
build_manifest_from_hf_split(DATASET_NAME, 'test', test_wavs_dir, test_manifest, LANG_TAG, LIMIT_TO_30_SECONDS)


Filter:   0%|          | 0/4378 [00:00<?, ? examples/s]

[train] Filtered 4378 -> 4243 examples (<= 30s)
[train] Wrote 4243 entries to /jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1/train_manifest.json


Filter:   0%|          | 0/542 [00:00<?, ? examples/s]

[validation] Filtered 542 -> 542 examples (<= 30s)
[validation] Wrote 542 entries to /jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1/dev_manifest.json


Filter:   0%|          | 0/928 [00:00<?, ? examples/s]

[test] Filtered 928 -> 926 examples (<= 30s)
[test] Wrote 926 entries to /jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1/test_manifest.json


'/jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1/test_manifest.json'

## Build the Static Noisy Eval Manifests

Mirrors the clean-vs-noisy WER methodology from your Whisper runs: a noisy copy of dev and test, built once with `augment_audio()`, so both models' noise-robustness gaps are comparable.

In [12]:
def build_noisy_manifest(clean_manifest_path: str, noisy_wavs_dir: str, noisy_manifest_path: str):
    """Read a clean manifest, write noisy copies of the audio, and a matching noisy manifest."""
    os.makedirs(noisy_wavs_dir, exist_ok=True)
    with open(clean_manifest_path) as fin, open(noisy_manifest_path, 'w') as fout:
        for i, line in enumerate(fin):
            entry = json.loads(line)
            audio_array, sr = sf.read(entry["audio_filepath"])
            noisy_array = augment_audio(np.asarray(audio_array, dtype=np.float32), sr, apply_prob=AUGMENT_PROB)

            noisy_path = os.path.join(noisy_wavs_dir, os.path.basename(entry["audio_filepath"]))
            sf.write(noisy_path, noisy_array, sr)

            noisy_entry = dict(entry)
            noisy_entry["audio_filepath"] = noisy_path
            noisy_entry["duration"] = len(noisy_array) / sr
            fout.write(json.dumps(noisy_entry) + "\n")

    print(f"Wrote {i + 1} noisy entries to {noisy_manifest_path}")
    return noisy_manifest_path


if USE_WAVEFORM_AUGMENTATION:
    noisy_dev_manifest  = build_noisy_manifest(dev_manifest, os.path.join(DATA_DIR, 'wavs', 'validation_noisy'),
                                                os.path.join(DATA_DIR, 'dev_manifest_noisy.json'))
    noisy_test_manifest = build_noisy_manifest(test_manifest, os.path.join(DATA_DIR, 'wavs', 'test_noisy'),
                                                os.path.join(DATA_DIR, 'test_manifest_noisy.json'))
else:
    noisy_dev_manifest = noisy_test_manifest = None
    print("USE_WAVEFORM_AUGMENTATION is False -- skipping noisy eval manifests.")


Wrote 542 noisy entries to /jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1/dev_manifest_noisy.json
Wrote 926 noisy entries to /jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1/test_manifest_noisy.json


## Configure the Fine-Tuning YAML

Loads NVIDIA's base streaming-prompt config and overrides it with the settings from the cells above, rather than passing a long chain of `++model.x.y=z` CLI overrides.

In [13]:
cfg = OmegaConf.load(BASE_CONFIG_PATH)

cfg.model.train_ds.manifest_filepath = train_manifest
cfg.model.train_ds.batch_size = BATCH_SIZE
cfg.model.train_ds.augmentor = OmegaConf.create(NEMO_AUGMENTOR_CONFIG) if NEMO_AUGMENTOR_CONFIG else None

cfg.model.validation_ds.manifest_filepath = dev_manifest
cfg.model.validation_ds.batch_size = EVAL_BATCH_SIZE

cfg.model.optim.lr = LEARNING_RATE
cfg.model.optim.weight_decay = WEIGHT_DECAY
if "sched" in cfg.model.optim:
    cfg.model.optim.sched.warmup_steps = LR_WARMUP_STEPS

cfg.trainer.max_epochs = MAX_EPOCHS
cfg.trainer.max_steps = MAX_STEPS
cfg.trainer.precision = PRECISION

cfg.exp_manager.exp_dir = OUTPUT_DIR
cfg.exp_manager.name = RUN_NAME
cfg.exp_manager.create_checkpoint_callback = True
cfg.exp_manager.checkpoint_callback_params.save_top_k = NUM_CHECKPOINTS_TO_STORE
cfg.exp_manager.checkpoint_callback_params.monitor = "val_wer"
cfg.exp_manager.checkpoint_callback_params.mode = "min"
cfg.exp_manager.create_early_stopping_callback = True
cfg.exp_manager.early_stopping_params = OmegaConf.create({
    "monitor": "val_wer",
    "mode": "min",
    "patience": EARLY_STOPPING_PATIENCE,
    "verbose": True,
})

RUN_CONFIG_PATH = os.path.join(DATA_DIR, "kenyan_english_finetune_config.yaml")
OmegaConf.save(cfg, RUN_CONFIG_PATH)
print(f"Wrote run config to: {RUN_CONFIG_PATH}")


Wrote run config to: /jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1/kenyan_english_finetune_config.yaml


## Run Fine-Tuning

This shells out to NeMo's `speech_to_text_finetune.py`, warm-starting from the downloaded base checkpoint via `init_from_nemo_model`. Expect this to run for a long time on a full dataset -- on Modal, prefer running this as a detached script over a long-lived function rather than blocking a notebook cell, and tail the logs instead.

In [14]:
train_cmd = [
    sys.executable, os.path.join(NEMO_DIR, "examples/asr/speech_to_text_finetune.py"),
    f"--config-path={DATA_DIR}",
    f"--config-name=kenyan_english_finetune_config.yaml",
    f"+init_from_nemo_model={HF_CKPT}",
]

print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)


Running: /usr/local/bin/python /jupyter_kernel/NeMo/examples/asr/speech_to_text_finetune.py --config-path=/jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1 --config-name=kenyan_english_finetune_config.yaml +init_from_nemo_model=/root/.cache/huggingface/hub/models--nvidia--nemotron-3.5-asr-streaming-0.6b/snapshots/f3d333391852ba876df169dcc9ba902d25b6ab0b/nemotron-3.5-asr-streaming-0.6b.nemo


OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


[NeMo I 2026-07-30 09:20:54 speech_to_text_finetune:198] Hydra config: name: FastConformer-Transducer-BPE-Prompt-Streaming
    model:
      sample_rate: 16000
      compute_eval_loss: false
      log_prediction: true
      skip_nan_grad: false
      model_defaults:
        enc_hidden: ${model.encoder.d_model}
        pred_hidden: 640
        joint_hidden: 640
        initialize_prompt_feature: true
        num_prompts: 128
        norm: None
        prompt_dictionary:
          en-US: 0
          en: 0
          en-GB: 1
          enGB: 1
          es-ES: 2
          esES: 2
          es-US: 3
          es: 3
          zh-CN: 4
          zh-ZH: 4
          zh-TW: 5
          hi-IN: 6
          hi: 6
          hi-HI: 6
          ar-AR: 7
          ar: 7
          fr-FR: 8
          fr: 8
          de-DE: 9
          de: 9
          ja-JP: 10
          ja-JA: 10
          ru-RU: 11
          ru: 11
          pt-BR: 12
          pt-PT: 13
          pt: 13
          ko-KR: 14
          ko:

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Error executing job with overrides: ['+init_from_nemo_model=/root/.cache/huggingface/hub/models--nvidia--nemotron-3.5-asr-streaming-0.6b/snapshots/f3d333391852ba876df169dcc9ba902d25b6ab0b/nemotron-3.5-asr-streaming-0.6b.nemo']
Traceback (most recent call last):
  File "/jupyter_kernel/NeMo/examples/asr/speech_to_text_finetune.py", line 201, in main
    exp_manager(trainer, cfg.get("exp_manager", None))
  File "/jupyter_kernel/NeMo/nemo/utils/exp_manager.py", line 594, in exp_manager
    cfg = OmegaConf.merge(schema, cfg)  # type: ExpManagerConfig
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
omegaconf.errors.ConfigKeyError: Key 'early_stopping_params' not in 'ExpManagerConfig'
    full_key: early_stopping_params
    object_type=ExpManagerConfig

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.


CalledProcessError: Command '['/usr/local/bin/python', '/jupyter_kernel/NeMo/examples/asr/speech_to_text_finetune.py', '--config-path=/jupyter_kernel/nemo_data/nemotron-3.5-asr-kenyan-english-nonstandard-robust_v1_run1', '--config-name=kenyan_english_finetune_config.yaml', '+init_from_nemo_model=/root/.cache/huggingface/hub/models--nvidia--nemotron-3.5-asr-streaming-0.6b/snapshots/f3d333391852ba876df169dcc9ba902d25b6ab0b/nemotron-3.5-asr-streaming-0.6b.nemo']' returned non-zero exit status 1.

## Locate the Best Checkpoint

In [ ]:
checkpoint_candidates = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, RUN_NAME, "**", "checkpoints", "*.nemo"), recursive=True)
)
assert checkpoint_candidates, f"No .nemo checkpoint found under {OUTPUT_DIR} -- check the training run's exp_manager output."
best_nemo_path = checkpoint_candidates[-1]
print(f"Using checkpoint: {best_nemo_path}")


## Evaluate: Clean vs Noisy WER

Uses NeMo's cache-aware streaming inference script (the same one used at deployment time) rather than a HF-style `.generate()` loop, since the encoder's cache-aware behavior is part of what's being evaluated.

In [ ]:
def transcript_normalizer(text: str) -> str:
    return " ".join(text.lower().split())


def run_streaming_eval(nemo_model_path: str, manifest_path: str, att_context_size, target_lang: str = "auto"):
    """Run NeMo's cache-aware streaming inference script and compute WER against the manifest's references."""
    out_dir = os.path.join(DATA_DIR, "eval_out", os.path.splitext(os.path.basename(manifest_path))[0])
    os.makedirs(out_dir, exist_ok=True)

    eval_cmd = [
        sys.executable, os.path.join(NEMO_DIR, "examples/asr/asr_cache_aware_streaming/speech_to_text_cache_aware_streaming_infer.py"),
        f"model_path={nemo_model_path}",
        f"dataset_manifest={manifest_path}",
        f"target_lang={target_lang}",
        f"att_context_size=\"{att_context_size}\"",
        "decoder_type=rnnt",
        "pad_and_drop_preencoded=true",
        f"batch_size={EVAL_BATCH_SIZE}",
        "cuda=0",
        "strip_lang_tags=false",
        f"output_path={out_dir}",
    ]
    print("Running:", " ".join(eval_cmd))
    subprocess.run(eval_cmd, check=True)

    # NeMo's streaming-inference script writes predictions back out alongside the manifest, one
    # prediction manifest per input; adjust this glob if your NeMo version names it differently.
    pred_manifests = glob.glob(os.path.join(out_dir, "*.json"))
    assert pred_manifests, f"No prediction output found in {out_dir} -- check the eval script's output_path convention for your NeMo version."

    references, predictions = [], []
    for pm in pred_manifests:
        with open(pm) as f:
            for line in f:
                entry = json.loads(line)
                references.append(entry["text"])
                predictions.append(entry.get("pred_text", entry.get("prediction", "")))

    refs_norm = [transcript_normalizer(r) for r in references]
    preds_norm = [transcript_normalizer(p) for p in predictions]
    wer = wer_metric.compute(references=refs_norm, predictions=preds_norm)
    return wer


clean_test_wer = run_streaming_eval(best_nemo_path, test_manifest, ATT_CONTEXT_SIZE, TARGET_LANG_AT_INFERENCE)
print(f"Clean test WER: {clean_test_wer:.4f}")

if noisy_test_manifest is not None:
    noisy_test_wer = run_streaming_eval(best_nemo_path, noisy_test_manifest, ATT_CONTEXT_SIZE, TARGET_LANG_AT_INFERENCE)
    print(f"Noisy test WER: {noisy_test_wer:.4f}")
    print(f"Clean/noisy gap: {(noisy_test_wer - clean_test_wer) * 100:.2f}pp")


## Save the Fine-Tuned Model

Copies the final `.nemo` checkpoint to a clearly named path. Push to the Hub the same way you did for the Whisper checkpoints, if you want to publish this one too.

In [ ]:
import shutil

FINAL_MODEL_PATH = os.path.join(BASE_DIR, f"{RUN_NAME}.nemo")
shutil.copy(best_nemo_path, FINAL_MODEL_PATH)
print(f"Saved final model to: {FINAL_MODEL_PATH}")

# from huggingface_hub import upload_file
# upload_file(
#     path_or_fileobj=FINAL_MODEL_PATH,
#     path_in_repo=os.path.basename(FINAL_MODEL_PATH),
#     repo_id="smainye/nemotron-3.5-asr-kenyan-english-nonstandard-v1",
#     repo_type="model",
# )
